# 3. Invalid Ballot Rate Map
อัตราบัตรเสียรายกลุ่มพิกัด — ช่วยระบุพื้นที่ที่ต้องลงให้ความรู้ก่อนเลือกตั้งครั้งหน้า

In [1]:
import sys
!{sys.executable} -m pip install folium -q

In [2]:
import pandas as pd
import folium
from pathlib import Path

CLEAN = Path('../cleaned/')
summary = pd.read_csv(CLEAN / 'master_summary_cleaned.csv')
coords  = pd.read_csv(CLEAN / 'coords_cleaned.csv')

JOIN = ['district', 'sub-district', 'unit_number']
s = summary[summary['unit_number'] != -1].copy()
df = s.merge(coords, on=JOIN, how='inner')

# aggregate รายจุดพิกัด (lat/lon เดียวกัน = กลุ่มหน่วยเดียวกัน)
agg = df.groupby(['district', 'latitude', 'longitude']).agg(
    total_ballots   = ('total_ballots',   'sum'),
    invalid_ballots = ('invalid_ballots', 'sum'),
    no_vote_ballots = ('no_vote_ballots', 'sum'),
    unit_count      = ('unit_number',     'nunique')
).reset_index()

agg['invalid_rate'] = (agg['invalid_ballots'] / agg['total_ballots'] * 100).round(2)
agg['novote_rate']  = (agg['no_vote_ballots']  / agg['total_ballots'] * 100).round(2)

def inv_color(r):
    if r >= 8:   return '#c0392b'
    elif r >= 6: return '#e74c3c'
    elif r >= 4: return '#f39c12'
    elif r >= 2: return '#f1c40f'
    else:        return '#2ecc71'

center = [agg['latitude'].mean(), agg['longitude'].mean()]
m = folium.Map(location=center, zoom_start=9, tiles='CartoDB positron')

for _, row in agg.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=9,
        color='white', weight=1,
        fill=True, fill_color=inv_color(row['invalid_rate']), fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>{row['district']}</b><br>"
            f"จำนวนหน่วย: {int(row['unit_count'])}<br>"
            f"บัตรทั้งหมด: {int(row['total_ballots']):,}<br>"
            f"บัตรเสีย: {int(row['invalid_ballots']):,} ({row['invalid_rate']}%)<br>"
            f"งดออกเสียง: {int(row['no_vote_ballots']):,} ({row['novote_rate']}%)",
            max_width=250
        ),
        tooltip=f"{row['district']} — บัตรเสีย {row['invalid_rate']}%"
    ).add_to(m)

legend_html = """
<div style="position:fixed;bottom:40px;left:40px;z-index:1000;background:white;
     padding:12px 16px;border-radius:8px;box-shadow:2px 2px 8px rgba(0,0,0,0.3);
     font-family:sans-serif;font-size:13px;line-height:1.8;">
  <b>อัตราบัตรเสีย</b><br>
  <span style="display:inline-block;width:14px;height:14px;background:#c0392b;border-radius:50%;margin-right:6px;vertical-align:middle;"></span>≥8% — วิกฤต<br>
  <span style="display:inline-block;width:14px;height:14px;background:#e74c3c;border-radius:50%;margin-right:6px;vertical-align:middle;"></span>6–8% — สูง<br>
  <span style="display:inline-block;width:14px;height:14px;background:#f39c12;border-radius:50%;margin-right:6px;vertical-align:middle;"></span>4–6% — ปานกลาง<br>
  <span style="display:inline-block;width:14px;height:14px;background:#f1c40f;border-radius:50%;margin-right:6px;vertical-align:middle;"></span>2–4% — ต่ำ<br>
  <span style="display:inline-block;width:14px;height:14px;background:#2ecc71;border-radius:50%;margin-right:6px;vertical-align:middle;"></span>&lt;2% — ดีมาก
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))

print('สรุปอัตราบัตรเสียรายอำเภอ:')
dist_summary = agg.groupby('district').apply(
    lambda x: pd.Series({
        'invalid_rate_avg': (x['invalid_ballots'].sum() / x['total_ballots'].sum() * 100).round(2),
        'total_units': int(x['unit_count'].sum()),
        'total_invalid': int(x['invalid_ballots'].sum())
    })
).reset_index()
print(dist_summary.sort_values('invalid_rate_avg', ascending=False).to_string(index=False))
print(f'\nอัตราบัตรเสียรวมทั้งเขต: {agg["invalid_ballots"].sum()/agg["total_ballots"].sum()*100:.2f}%')

m


สรุปอัตราบัตรเสียรายอำเภอ:
  district  invalid_rate_avg  total_units  total_invalid
เมืองลำปาง              5.96         31.0         1558.0
  เมืองปาน              5.42         65.0         2865.0
    แจ้ห่ม              5.13         72.0         3100.0
  วังเหนือ              4.94         84.0         3307.0
       งาว              4.16         90.0         3376.0

อัตราบัตรเสียรวมทั้งเขต: 4.94%


/var/folders/kd/fgxw73j51n56v52rqjz_9xkw0000gn/T/ipykernel_8392/736488617.py:65: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dist_summary = agg.groupby('district').apply(
